# Daily Bank KPI View
create Daily Bank KPI view with the columns
- txn_date
- total_customers
- total_accounts
- total_balance
- total_transactions
- total_transaction_amount
- avg_credit_score
- high_risk_customers

In [0]:
CREATE OR REPLACE VIEW neo_bank.gold.vw_daily_bank_kpi AS

WITH customers_agg AS (
    select count(customer_id) as total_customers
    from neo_bank.gold.dim_customers
),

accounts_agg AS (
    select 
        count(account_id) as total_accounts,
        sum(balance) as total_balance
    from neo_bank.gold.dim_accounts
),

txn_agg AS (
    select 
        count(txn_id) as total_transactions,
        sum(amount) as total_transaction_amount,
        full_date as txn_date
    from neo_bank.gold.fact_transactions 
    inner join neo_bank.gold.dim_date 
    on neo_bank.gold.fact_transactions.txn_date_key == neo_bank.gold.dim_date.date_key
    group by txn_date order by txn_date
),

credit_report_agg AS (
    select 
        avg(credit_score) as avg_credit_score,
        sum(CASE WHEN risk_grade=='HIGH' THEN 1 ELSE 0 END) as high_risk_customers
    from neo_bank.gold.fact_credit_bureau_reports
)

select
    txn_date,
    total_customers,
    total_accounts,
    total_balance,
    total_transactions,
    total_transaction_amount,
    avg_credit_score,
    high_risk_customers

from customers_agg 
CROSS JOIN accounts_agg
CROSS JOIN txn_agg
CROSS JOIN credit_report_agg
order by txn_date